# Hari Gradio! (Membangun Antarmuka Pengguna untuk AI)

Gradio memungkinkan kita untuk mengubah skrip Python sederhana, terutama model Machine Learning dan API LLM, menjadi aplikasi web interaktif hanya dengan beberapa baris kode.


## 1. Persiapan dan Import Modul
Langkah pertama dan paling krusial adalah mengimpor semua pustaka (library) yang dibutuhkan. Pada bagian ini kita mengimpor:
* `os`: Modul bawaan Python untuk berinteraksi dengan sistem operasi, khususnya untuk membaca variabel lingkungan (environment variables).
* `dotenv`: Pustaka pihak ketiga untuk memuat variabel rahasia (seperti API Keys) dari file `.env` ke dalam sistem. Ini adalah praktik keamanan standar agar *key* Anda tidak tertulis langsung di dalam kode.
* `OpenAI`: Kelas utama dari pustaka `openai` resmi untuk berinteraksi dengan API OpenAI (dan API lain yang kompatibel).

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## 2. Mengimpor Gradio
Di sini kita mengimpor pustaka utama kita hari ini, yaitu Gradio. Biasanya, Gradio diimpor dengan alias `gr` untuk mempersingkat penulisan saat kita memanggil fungsi-fungsinya di sel-sel berikutnya.

In [5]:
import gradio as gr 

## 3. Memuat Kunci API (API Keys)
Kode di bawah ini akan mencari file bernama `.env` di folder yang sama dengan notebook ini dan memuat isinya. File `.env` ini seharusnya berisi kunci rahasia Anda, yaitu `OPENAI_API_KEY`.

Kode ini juga menyertakan *print statement* sederhana sebagai proses *debugging* untuk memastikan API Key sudah terbaca oleh sistem tanpa menampilkannya secara utuh.

In [6]:
# Memuat variabel lingkungan dari file bernama .env
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key berhasil dimuat dan dimulai dengan: {openai_api_key[:8]}...")
else:
    print("Peringatan: OpenAI API Key tidak ditemukan. Pastikan file .env sudah benar.")

OpenAI API Key berhasil dimuat dan dimulai dengan: sk-proj-...


## 4. Inisialisasi Klien OpenAI
Kita membuat instansiasi klien OpenAI. Secara otomatis, fungsi `OpenAI()` akan mencari variabel `OPENAI_API_KEY` di dalam *environment* sistem Anda dan menggunakannya untuk autentikasi koneksi ke server ChatGPT.

In [7]:
# Menghubungkan ke server OpenAI
openai = OpenAI()

## 5. Membungkus Panggilan LLM dalam Sebuah Fungsi
Gradio beroperasi dengan cara memanggil fungsi Python setiap kali pengguna menekan tombol "Submit". Oleh karena itu, kita harus membungkus interaksi API ke dalam sebuah fungsi standar. 

Fungsi `message_gpt(prompt)` di bawah ini bertugas menerima teks dari pengguna (`prompt`), menggabungkannya dengan `system_message`, mengirimkannya ke model `gpt-4.1-mini`, dan mengembalikan balasan berupa teks murni.

In [8]:
# Mari kita bungkus panggilan ke model gpt-4.1-mini dalam fungsi sederhana

system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message}, 
        {"role": "user", "content": prompt}
    ]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

## 6. Uji Coba Fungsi LLM
Sebelum memasukkannya ke UI Gradio, sangat disarankan untuk menguji fungsi tersebut secara langsung guna memastikan koneksi internet dan API Key OpenAI Anda berfungsi dengan baik.

In [9]:
# Menanyakan tanggal hari ini bisa mengungkapkan "training cut off" (batas data pelatihan) dari model

message_gpt("What is today's date?")

"Today's date is June 9, 2024."

---
## Waktunya Antarmuka Pengguna (User Interface)!
Mulai dari titik ini, kita akan menggunakan Gradio. Konsep dasar Gradio sangat logis: Anda mendefinisikan sebuah fungsi Python (`fn`), menentukan tipe input yang akan diberikan pengguna (`inputs`), dan menentukan di mana hasil fungsi tersebut akan ditampilkan (`outputs`).

## 7. Fungsi Simulasi (Dummy Function)
Sebagai langkah awal untuk memahami Gradio tanpa intervensi API yang memakan waktu, kita buat fungsi sederhana bernama `shout(text)`. Fungsi ini tidak menggunakan AI, ia hanya menerima string, mencetaknya di terminal, dan mengembalikannya dalam HURUF KAPITAL menggunakan method `.upper()`.

In [10]:
def shout(text):
    print(f"Fungsi shout dipanggil dengan input: {text}")
    return text.upper()

In [11]:
shout("hello")

Fungsi shout dipanggil dengan input: hello


'HELLO'

## 8. Menjalankan Aplikasi Gradio Pertama
Ini adalah wujud Gradio yang paling mendasar:
* `gr.Interface`: Pembuat UI utama.
* `fn=shout`: Fungsi yang telah kita buat sebelumnya.
* `inputs="textbox"`: Kita memberitahu Gradio untuk menyediakan sebuah kotak teks bagi pengguna untuk mengetik.
* `outputs="textbox"`: Hasil kembalian (return) dari fungsi `shout` akan ditampilkan di kotak teks lainnya.
* `flagging_mode="never"`: Menghilangkan tombol "Flag" yang secara default ada di Gradio.
* `.launch()`: Menjalankan server web lokal.

In [12]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### CATATAN: Menggunakan fitur Share Gradio

Gradio memiliki fitur untuk membuat tautan publik (public link) agar aplikasi Anda bisa diakses oleh siapa saja di internet selama 72 jam menggunakan parameter share=True.

Namun, banyak program Antivirus dan jaringan perusahaan yang memblokir teknologi ini karena alasan keamanan. Jika sel kode berikutnya mengalami error, lewati saja dan gunakan versi lokal.

In [15]:
# share=True berarti aplikasi ini bisa diakses dari internet secara publik
# Jika Anda menggunakan jaringan kantor yang ketat, sel ini mungkin akan menghasilkan error.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 9. Membuka Otomatis di Tab Browser
Parameter `inbrowser=True` sangat berguna saat Anda bekerja di lokal. Ia akan memerintahkan OS untuk membuka tab baru di *browser* bawaan Anda secara otomatis begitu server Gradio berjalan.

In [16]:
# inbrowser=True akan otomatis membuka tab localhost baru di browser Anda

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## 10. Menambahkan Autentikasi Login

Gradio mempermudah pembuatan sistem pelindung menggunakan *username* dan *password*. Anda hanya perlu menambahkan parameter `auth=("username", "password")`.

Tentu saja, dalam skenario nyata dunia kerja (production), jangan pernah mengetikkan password langsung di kode (*hardcode*). Gunakanlah variabel lingkungan dari file `.env`!

In [18]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## 11. Memaksa Mode Gelap (Dark Mode)

Secara bawaan, Gradio menyesuaikan warnanya dengan preferensi sistem operasi pengguna (Light atau Dark). Walaupun Gradio merekomendasikan untuk membiarkannya demi aksesibilitas pengguna, ada kalanya Anda ingin memaksakan desain *Dark Mode*.

Cara kerjanya adalah dengan menyisipkan sekumpulan kode JavaScript sederhana (`js=force_dark_mode`) yang akan menambahkan parameter `__theme=dark` ke dalam URL secara otomatis saat halaman dimuat.

In [19]:
# Definisikan skrip JS ini lalu masukkan ke parameter js pada gr.Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

/home/khai/LLMEng2/AITF/lib/python3.12/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: js. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## 12. Kustomisasi Komponen Input dan Output
Menggunakan string `"textbox"` memang mudah, tapi sangat terbatas. Di sini, kita mulai membuat objek komponen Gradio sesungguhnya (yakni `gr.Textbox()`). 

Dengan menggunakan objek, kita bisa memberikan kustomisasi lanjutan, seperti:
* `label`: Judul di atas kotak teks.
* `info`: Teks bantuan kecil di bawah label.
* `lines`: Berapa jumlah baris teks yang ditampilkan secara *default*.

Kita juga menambahkan parameter `examples`, yang akan memunculkan tombol-tombol berisi teks prasetel yang bisa diklik pengguna agar mereka tidak perlu mengetik dari awal.

In [20]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout App", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## 13. Integrasi Gradio dengan Fungsi ChatGPT
Sekarang, kita gabungkan antarmuka yang sudah dipercantik tadi dengan fungsi kecerdasan buatan sesungguhnya yang telah kita buat di awal sesi, yaitu `message_gpt`.

In [21]:
# Mengganti fungsi "shout" dengan fungsi "message_gpt" yang memanggil OpenAI API

message_input = gr.Textbox(label="Your message:", info="Enter a message for ChatGPT", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="ChatGPT Sederhana", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


## 14. Menyempurnakan Tampilan dengan Komponen Markdown
Model LLM sangat sering menggunakan format Markdown saat membalas (misalnya menggunakan `**` untuk huruf tebal, atau `-` untuk daftar). Jika kita memakai `gr.Textbox` untuk outputnya, format itu akan terlihat mentah dan berantakan. Solusinya adalah menggunakan komponen `gr.Markdown()` sebagai `outputs`.

> *Catatan:* Di sini kita mengubah variabel global `system_message` agar secara spesifik menginstruksikan ChatGPT untuk membalas dalam format Markdown tanpa menggunakan blok kode yang panjang.

In [23]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for ChatGPT", lines=7)
message_output = gr.Markdown(label="Response:") # Output sekarang menggunakan Markdown

view = gr.Interface(
    fn=message_gpt,
    title="ChatGPT dengan Markdown", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## 15. Membangun Fitur Streaming (Penting untuk UX)
Ketika LLM menghasilkan teks yang panjang, pengguna bisa merasa aplikasi Anda "hang" jika mereka harus menunggu seluruh teks selesai dibuat. 

Fitur **Streaming** mengatasi ini dengan menampilkan teks kata demi kata, persis seperti antarmuka ChatGPT asli. Di Python, kita tidak menggunakan `return` untuk ini, melainkan memanfaatkan konsep **Generators** dengan kata kunci `yield`. Setiap kali potongan teks baru (`chunk`) tiba dari API OpenAI, `yield` akan langsung melempar teks tersebut ke antarmuka Gradio untuk ditampilkan secara *real-time*.

In [24]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    
    # Menambahkan stream=True ke konfigurasi OpenAI
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    
    result = ""
    for chunk in stream:
        # Menangkap potongan teks dan menambahkannya ke hasil akhir
        result += chunk.choices[0].delta.content or ""
        # Mengembalikan data ke UI secara bertahap
        yield result

## 16. UI Gradio yang Mendukung Streaming
Gradio dirancang sangat cerdas. Anda tidak perlu mengubah satupun kode UI Anda! Jika fungsi `fn` yang Anda berikan adalah sebuah *Generator* (mengandung `yield`), Gradio secara otomatis akan memahami bahwa itu adalah aliran data berkelanjutan (*stream*) dan akan memperbarui layar secara instan.

In [25]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for ChatGPT", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt, # Memanggil fungsi stream_gpt yang memiliki 'yield'
    title="ChatGPT Streaming", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## 17. Menggabungkan Semuanya (Pemilihan Versi Model)

Kita akan membuat aplikasi kita lebih dinamis dengan memungkinkan pengguna memilih versi model OpenAI mana yang ingin mereka gunakan (misalnya `gpt-4.1-mini` untuk respons cepat atau `gpt-4o` untuk analitik yang lebih berat).

Kita membuat fungsi *wrapper* bernama `stream_openai_model`. Fungsi ini menerima parameter tambahan yaitu `model_name` yang nilainya akan didapatkan dari komponen UI *Dropdown*.

In [26]:
def stream_openai_model(prompt, model_name):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    
    # Menggunakan model_name yang dipilih oleh pengguna
    stream = openai.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

## 18. UI dengan Input Majemuk (Textbox dan Dropdown)
Untuk menggunakan pilihan model, parameter `inputs` kita sekarang berisi lebih dari satu elemen (*List*). 
* Elemen pertama adalah `gr.Textbox` untuk prompt teks.
* Elemen kedua adalah `gr.Dropdown` untuk memilih versi model OpenAI. 

Urutan elemen di dalam list `inputs` ini *sangat penting* karena nilainya akan dilemparkan secara berurutan sebagai argumen pertama (`prompt`) dan argumen kedua (`model_name`) ke fungsi `stream_openai_model`.

In [27]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["gpt-4.1-mini", "gpt-4o"], label="Pilih Versi Model OpenAI", value="gpt-4.1-mini")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_openai_model,
    title="OpenAI Multi-Model Explorer", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
        ["Explain the Transformer architecture to a layperson", "gpt-4.1-mini"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "gpt-4o"]
    ], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


---
# Proyek Akhir: Generator Brosur Perusahaan



## 1. Persiapan dan Import Modul
Langkah pertama dan paling krusial adalah mengimpor semua pustaka (library) yang dibutuhkan.
* `os` & `dotenv`: Untuk memuat variabel rahasia (seperti API Keys) dari file `.env` ke dalam sistem agar aman.
* `OpenAI`: Kelas utama untuk berinteraksi dengan API ChatGPT.
* `gradio`: Pustaka utama kita untuk membuat UI.

In [32]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

## 2. Memuat Kunci API dan Inisialisasi OpenAI
Kode di bawah ini akan memuat file `.env` Anda yang berisi `OPENAI_API_KEY`. Setelah berhasil dimuat, kita langsung membuat instansiasi klien OpenAI.

In [33]:
# Memuat variabel lingkungan dari file .env
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key berhasil dimuat: {openai_api_key[:8]}...")
else:
    print("Peringatan: OpenAI API Key tidak ditemukan.")

# Menghubungkan ke server OpenAI
openai = OpenAI()

OpenAI API Key berhasil dimuat: sk-proj-...


## 3. Fungsi LLM Dasar (Tanpa Streaming)
Kita mulai dengan fungsi sederhana yang menerima teks, mengirimkannya ke model `gpt-4.1-mini`, dan mengembalikan jawabannya.

In [34]:
system_message_dasar = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message_dasar}, 
        {"role": "user", "content": prompt}
    ]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

## 4. UI Gradio Pertama Anda (Level: Pemula)
Ini adalah wujud Gradio yang paling mendasar menggunakan `gr.Interface`. Anda hanya perlu menentukan fungsi `fn`, bentuk `inputs`, dan bentuk `outputs`. Gradio akan merakit tampilannya secara otomatis.

In [41]:
message_input = gr.Textbox(label="Pesan Anda:", info="Ketik sesuatu untuk ChatGPT", lines=5)
message_output = gr.Markdown(label="Balasan AI:") 

view = gr.Interface(
    fn=message_gpt,
    title="ChatGPT Interface Sederhana", 
    inputs=[message_input], 
    outputs=[message_output], 
    flagging_mode="never"
)

# Menjalankan aplikasi di tab browser baru
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


## 5. Membangun Fitur Streaming (Level: Menengah)
Menunggu jawaban panjang dari LLM bisa membosankan. Fitur **Streaming** mengatasi ini dengan menampilkan teks kata demi kata. 

Di Python, kita menggunakan **Generators** dengan kata kunci `yield`. Setiap kali potongan teks (`chunk`) tiba dari OpenAI, `yield` akan langsung melempar teks tersebut ke layar.

In [36]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that responds in markdown."},
        {"role": "user", "content": prompt}
    ]
    
    # Tambahkan stream=True
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result # <-- Mengirim data ke layar secara real-time

## 6. UI dengan Pemilih Model (Multi-Input)
Kita buat antarmuka di mana parameter `inputs` berisi daftar (*List*) komponen. Elemen pertama untuk `prompt`, elemen kedua untuk memilih versi model OpenAI.

In [42]:
def stream_openai_model(prompt, model_name):
    messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": prompt}]
    stream = openai.chat.completions.create(model=model_name, messages=messages, stream=True)
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

# UI Setup
input_teks = gr.Textbox(label="Pesan Anda:")
input_model = gr.Dropdown(["gpt-4.1-mini", "gpt-4o"], label="Pilih Versi Model", value="gpt-4.1-mini")
output_teks = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_openai_model,
    title="OpenAI Streaming & Model Selector", 
    inputs=[input_teks, input_model], 
    outputs=[output_teks], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


---
# 🎓 PROYEK AKHIR: Generator Soal Ujian AI (Level: Ahli)

Sekarang kita masuk ke bagian yang **menantang**. Kita akan meninggalkan `gr.Interface` yang otomatis tapi kaku, dan beralih menggunakan **`gr.Blocks()`**. 

`gr.Blocks()` mengizinkan kita mendesain *layout* (tata letak) sesuka hati layaknya membuat *website* sungguhan. Kita bisa menaruh tombol di samping teks, membagi layar menjadi dua kolom, dan banyak lagi.

**Skenario Proyek:** Kita akan membuat aplikasi pembuat soal otomatis (misal untuk gaya ujian sekolah atau UTBK). Pengguna memasukkan teks/materi, memilih tingkat kesulitan, menentukan jumlah soal, dan AI akan meracik soal pilihan ganda lengkap dengan kunci jawaban.

## 7. Setup System Prompt Proyek Akhir
Kita instruksikan ChatGPT untuk berperan sebagai Guru Pembuat Soal yang sangat teliti.

In [38]:
guru_system_message = """
Anda adalah seorang pembuat soal ujian profesional yang ahli.
Tugas Anda adalah membaca teks materi yang diberikan oleh pengguna, kemudian membuat soal pilihan ganda (A, B, C, D, E).

Aturan output:
1. Berikan format soal yang rapi menggunakan Markdown.
2. Selalu sertakan 'Kunci Jawaban & Pembahasan' secara detail di bagian paling bawah setelah semua soal selesai.
3. Sesuaikan gaya dan kedalaman pertanyaan dengan tingkat kesulitan yang diminta pengguna.
"""

## 8. Fungsi Logika Utama (Multi-Parameter Prompting)
Fungsi ini akan menerima 3 parameter berbeda dari UI: teks materi, tingkat kesulitan, dan jumlah soal. Kita akan merakit ketiganya menjadi satu *Super Prompt* sebelum dikirim ke OpenAI.

In [39]:
def generate_quiz(materi, tingkat_kesulitan, jumlah_soal):
    # Mengosongkan UI sesaat ketika tombol diklik
    yield "Menganalisis materi dan membuat soal... Mohon tunggu ⏳"
    
    # Merakit Prompt yang kompleks
    instruksi_user = f"""
    Buatkan {jumlah_soal} soal pilihan ganda dengan tingkat kesulitan: {tingkat_kesulitan}.
    
    Sumber Materi Pelajaran:
    \"\"\"{materi}\"\"\"
    """
    
    messages = [
        {"role": "system", "content": guru_system_message},
        {"role": "user", "content": instruksi_user}
    ]
    
    # Memanggil API dengan mode streaming
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )
    
    hasil_akhir = ""
    for chunk in stream:
        hasil_akhir += chunk.choices[0].delta.content or ""
        yield hasil_akhir # Merender output bertahap

## 9. Membangun Custom UI dengan `gr.Blocks()`
Di sinilah letak keseruannya! Perhatikan bagaimana kita menyusun komponen:
1. `with gr.Blocks() as demo:` -> Inisialisasi kanvas kosong.
2. `with gr.Row():` -> Membuat baris horizontal.
3. `with gr.Column():` -> Membagi baris tersebut menjadi kolom vertikal.
4. Kita menyambungkan aksi klik tombol dengan fungsi `btn.click(fn=..., inputs=..., outputs=...)` secara manual. Ini memberi kita kontrol 100% atas interaksi komponen.

In [43]:
# Membuka kanvas Blocks
with gr.Blocks(theme=gr.themes.Soft()) as quiz_app:
    
    # Bagian Header / Judul
    gr.Markdown("# AI Quiz & Exam Generator")
    gr.Markdown("Masukkan materi pelajaran di bawah ini, dan AI akan otomatis membuatkan soal latihan beserta pembahasannya.")
    
    # Membagi layar menjadi 2 kolom utama (kiri untuk Input, kanan untuk Output)
    with gr.Row():
        
        # --- KOLOM KIRI: Area Input ---
        with gr.Column(scale=1): 
            input_materi = gr.Textbox(
                label="Tempel Teks Materi Pelajaran Di Sini", 
                lines=8, 
                placeholder="Contoh: Proses fotosintesis pada tumbuhan terjadi di kloroplas..."
            )
            
            # Membagi input pengaturan menjadi 1 baris sejajar
            with gr.Row():
                input_tingkat = gr.Dropdown(
                    ["Mudah", "Sedang", "Sulit", "HOTS (Gaya UTBK)"], 
                    label="Tingkat Kesulitan", 
                    value="Sedang"
                )
                input_jumlah = gr.Slider(
                    minimum=1, maximum=10, step=1, value=3, 
                    label="Jumlah Soal"
                )
                
            # Tombol eksekusi utama
            btn_generate = gr.Button("Buat Soal Sekarang", variant="primary")
            
        # --- KOLOM KANAN: Area Output ---
        with gr.Column(scale=1):
            output_soal = gr.Markdown(label="Lembar Soal & Pembahasan", value="Hasil akan muncul di sini...")
            
    # Menyambungkan Tombol dengan Fungsi Python
    # Saat 'btn_generate' diklik, jalankan 'generate_quiz' dengan mengambil nilai dari ke-3 input
    btn_generate.click(
        fn=generate_quiz, 
        inputs=[input_materi, input_tingkat, input_jumlah], 
        outputs=[output_soal]
    )

# Meluncurkan aplikasi Blocks
quiz_app.launch(inbrowser=True)

/tmp/ipykernel_143797/4055485056.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as quiz_app:


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


## Kesimpulan Proyek Akhir
Dengan menggunakan `gr.Blocks()`, Anda baru saja memisahkan antara bagian *Input Parameter* (Kiri) dan bagian *Output Hasil* (Kanan). Struktur ini jauh lebih profesional dan *User-Friendly* dibandingkan `gr.Interface` biasa yang menumpuk semuanya ke bawah.

Pola arsitektur Gradio seperti inilah yang digunakan oleh para *AI Engineer* untuk membangun alat bantu (*tools*) pemrosesan teks, generator dokumen, atau dasbor analitik internal di dunia kerja nyata!